# Week 5: Error Analysis & Feature Attribution
**Depends on:** Week 1, Week 3, and Week 4 result files.

**This notebook:**
1. Loads all result CSVs (Week 3 and Week 4)
2. Error analysis - where do models fail and why
3. Agreement analysis - which rows all models get right vs wrong
4. Feature attribution - RF and XGBoost feature importances
5. Accuracy breakdown by occupation and age group
6. Manager removal experiment - does removing the most ambiguous occupation improve accuracy
7. Final summary table across all models

**System prompt fix:** Updated to include associates degree in college definition.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

sns.set_theme(style="whitegrid")

RESULTS_DIR = "../results"
DATA_DIR    = "../data"
label_map   = {"college": 1, "not_college": 0}

def get_metrics(df, pred_col, true_col="true_label"):
    valid  = df[df[pred_col].isin(["college", "not_college"])].copy()
    y_true = valid[true_col].map(label_map)
    y_pred = valid[pred_col].map(label_map)
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "auc_roc":  round(roc_auc_score(y_true, y_pred), 4),
        "n":        len(valid),
    }

print("Imports OK")

## 2. Load All Result Files

In [ ]:
w3_zs = pd.read_csv(f"{RESULTS_DIR}/week3_zeroshot_raw.csv")
w3_fs = pd.read_csv(f"{RESULTS_DIR}/week3_fewshot_raw.csv")
w4_zs = pd.read_csv(f"{RESULTS_DIR}/week4_zeroshot_raw.csv")
w4_fs = pd.read_csv(f"{RESULTS_DIR}/week4_fewshot_raw.csv")
w4_nt = pd.read_csv(f"{RESULTS_DIR}/week4_nothink_raw.csv")

df_train = pd.read_csv(f"{DATA_DIR}/week3_train_5000.csv")
df_test  = pd.read_csv(f"{DATA_DIR}/week3_test_5000.csv")

print(f"Week 3 zero-shot:  {len(w3_zs)} rows")
print(f"Week 3 few-shot:   {len(w3_fs)} rows")
print(f"Week 4 zero-shot:  {len(w4_zs)} rows")
print(f"Week 4 few-shot:   {len(w4_fs)} rows")
print(f"Week 4 no-think:   {len(w4_nt)} rows")
print(f"Train:             {len(df_train)} rows")
print(f"Test:              {len(df_test)} rows")

## 3. Build Unified Error Table

In [ ]:
unified = w4_zs[["row_id", "input", "occupation", "audit_verdict",
                  "verdict_num", "true_label"]].copy()

unified = unified.merge(
    df_test[["occupation", "age", "sex", "marital_status", "state"]].reset_index(),
    left_on="row_id", right_on="index", how="left", suffixes=("", "_demo")
)

unified["w3_zs_pred"] = w3_zs["pred_label"].values[:len(unified)]
unified["w3_fs_pred"] = w3_fs["pred_label"].values[:len(unified)]
unified["w4_zs_pred"] = w4_zs["pred_label"].values
unified["w4_fs_pred"] = w4_fs["pred_label"].values
unified["w4_nt_pred"] = w4_nt["pred_label"].values

for col, pred in [
    ("w3_zs_correct", "w3_zs_pred"),
    ("w3_fs_correct", "w3_fs_pred"),
    ("w4_zs_correct", "w4_zs_pred"),
    ("w4_fs_correct", "w4_fs_pred"),
    ("w4_nt_correct", "w4_nt_pred"),
]:
    unified[col] = unified[pred] == unified["true_label"]

correct_cols = ["w3_zs_correct", "w3_fs_correct", "w4_zs_correct",
                "w4_fs_correct", "w4_nt_correct"]
unified["models_correct"] = unified[correct_cols].sum(axis=1)

print(f"Unified table: {len(unified)} rows")
print(f"\nModels correct distribution:")
print(unified["models_correct"].value_counts().sort_index())

## 4. Agreement Analysis

In [ ]:
all_correct  = unified[unified["models_correct"] == 5]
all_wrong    = unified[unified["models_correct"] == 0]
disagreement = unified[(unified["models_correct"] > 0) & (unified["models_correct"] < 5)]

print(f"All 5 models correct:  {len(all_correct):3d} rows ({len(all_correct)/len(unified)*100:.1f}%)")
print(f"All 5 models wrong:    {len(all_wrong):3d} rows ({len(all_wrong)/len(unified)*100:.1f}%)")
print(f"Mixed results:         {len(disagreement):3d} rows ({len(disagreement)/len(unified)*100:.1f}%)")

print(f"\n=== ALL MODELS WRONG - Top occupations ===")
print(all_wrong["occupation"].value_counts().head(10))

print(f"\n=== ALL MODELS WRONG - Audit verdict ===")
print(all_wrong["audit_verdict"].value_counts())

print(f"\n=== ALL MODELS WRONG - Sample rows ===")
for _, row in all_wrong.head(5).iterrows():
    print(f"  {row['input']}")
    print(f"  True: {row['true_label']} | Verdict: {row['audit_verdict']}")
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = unified["models_correct"].value_counts().sort_index()
colors_agree = ["#D85A30", "#EF9F27", "#EF9F27", "#EF9F27", "#EF9F27", "#1D9E75"]
axes[0].bar(vc.index, vc.values, color=colors_agree[:len(vc)])
axes[0].set_title("Model Agreement Distribution")
axes[0].set_xlabel("Number of models correct (out of 5)")
axes[0].set_ylabel("Number of rows")
for i, (x, y) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(x, y + 1, str(y), ha="center", fontweight="bold")

verdict_agree = unified.groupby("audit_verdict")["models_correct"].mean().reindex(["yes", "no", "maybe"])
axes[1].bar(verdict_agree.index, verdict_agree.values, color=["#1D9E75", "#378ADD", "#EF9F27"])
axes[1].set_title("Avg Models Correct by Audit Verdict")
axes[1].set_xlabel("Audit verdict")
axes[1].set_ylabel("Avg models correct (out of 5)")
axes[1].set_ylim(0, 5)
for i, (x, y) in enumerate(zip(verdict_agree.index, verdict_agree.values)):
    axes[1].text(i, y + 0.05, f"{y:.2f}", ha="center", fontweight="bold")

plt.suptitle("Model Agreement Analysis", fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_agreement.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_agreement.png")

## 5. Error Analysis by Occupation

In [ ]:
occ_acc = unified.groupby("occupation").agg(
    size=("true_label", "count"),
    w4_nt_acc=("w4_nt_correct", "mean"),
    w4_zs_acc=("w4_zs_correct", "mean"),
    w3_fs_acc=("w3_fs_correct", "mean"),
    college_rate=("true_label", lambda x: (x == "college").mean()),
    audit_verdict=("audit_verdict", "first")
).reset_index().sort_values("w4_nt_acc")

print("=== HARDEST occupations for 30B reasoning OFF ===")
print(occ_acc.head(10)[["occupation", "size", "w4_nt_acc", "college_rate", "audit_verdict"]].to_string(index=False))

print("\n=== EASIEST occupations for 30B reasoning OFF ===")
print(occ_acc.tail(10)[["occupation", "size", "w4_nt_acc", "college_rate", "audit_verdict"]].to_string(index=False))

In [ ]:
top_occs = unified["occupation"].value_counts().head(20).index
occ_plot = occ_acc[occ_acc["occupation"].isin(top_occs)].sort_values("w4_nt_acc")
occ_plot["occ_clean"] = occ_plot["occupation"].str.replace("_", " ").str.strip()

color_map  = {"yes": "#1D9E75", "no": "#378ADD", "maybe": "#EF9F27"}
bar_colors = [color_map.get(v, "gray") for v in occ_plot["audit_verdict"]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(occ_plot["occ_clean"], occ_plot["w4_nt_acc"], color=bar_colors)
ax.axvline(x=unified["w4_nt_correct"].mean(), color="red", linestyle="--",
           label=f'Overall ({unified["w4_nt_correct"].mean():.1%})')
ax.set_title("30B Reasoning OFF - Accuracy by Occupation\n(green=yes, blue=no, orange=maybe)", fontsize=11)
ax.set_xlabel("Accuracy")
ax.set_xlim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_accuracy_by_occupation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_accuracy_by_occupation.png")

## 6. Error Analysis by Age Group

In [ ]:
unified["age_group"] = pd.cut(
    unified["age"],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
)

age_acc = unified.groupby("age_group", observed=True).agg(
    size=("true_label", "count"),
    w3_zs=("w3_zs_correct", "mean"),
    w3_fs=("w3_fs_correct", "mean"),
    w4_zs=("w4_zs_correct", "mean"),
    w4_nt=("w4_nt_correct", "mean"),
    college_rate=("true_label", lambda x: (x == "college").mean())
).reset_index()

print("=== ACCURACY BY AGE GROUP ===")
print(age_acc.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(age_acc))
width = 0.2
ax.bar([i - 1.5*width for i in x], age_acc["w3_zs"], width, label="4B zero-shot",  color="#378ADD", alpha=0.85)
ax.bar([i - 0.5*width for i in x], age_acc["w3_fs"], width, label="4B few-shot",   color="#1D9E75", alpha=0.85)
ax.bar([i + 0.5*width for i in x], age_acc["w4_zs"], width, label="30B zero-shot", color="#EF9F27", alpha=0.85)
ax.bar([i + 1.5*width for i in x], age_acc["w4_nt"], width, label="30B OFF",       color="#8B5CF6", alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels([f"{g}\n({s} rows)" for g, s in zip(age_acc["age_group"], age_acc["size"])])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Accuracy by Age Group - All LLM Configurations", fontsize=12)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_accuracy_by_age.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_accuracy_by_age.png")

## 7. Feature Attribution: RF and XGBoost

In [ ]:
FEATURES = ["age", "sex", "marital_status", "occupation", "state"]

X_train = df_train[FEATURES].copy()
y_train = df_train["label"].copy()
X_test  = df_test[FEATURES].copy()
y_test  = df_test["label"].copy()

label_encoders = {}
for col in ["sex", "marital_status", "occupation", "state"]:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col]  = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=6,
    random_state=42, eval_metric="logloss", verbosity=0
)
xgb_model.fit(X_train, y_train)

rf_imp  = pd.Series(rf.feature_importances_,  index=FEATURES).sort_values(ascending=False)
xgb_imp = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)

print("RF Feature Importances:")
print(rf_imp)
print("\nXGBoost Feature Importances:")
print(xgb_imp)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rf_imp.plot(kind="bar", ax=axes[0], color="#378ADD", edgecolor="white")
axes[0].set_title("Random Forest - Feature Importances")
axes[0].set_ylabel("Importance")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(rf_imp.values):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=9)

xgb_imp.plot(kind="bar", ax=axes[1], color="#1D9E75", edgecolor="white")
axes[1].set_title("XGBoost - Feature Importances")
axes[1].set_ylabel("Importance")
axes[1].tick_params(axis="x", rotation=20)
for i, v in enumerate(xgb_imp.values):
    axes[1].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=9)

plt.suptitle("Feature Importances: RF vs XGBoost", fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_feature_importances.png")

## 8. LLM vs XGBoost: Where These Models Disagree

In [ ]:
rf_pred  = rf.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

unified["rf_pred"]     = ["college" if p == 1 else "not_college" for p in rf_pred]
unified["xgb_pred"]    = ["college" if p == 1 else "not_college" for p in xgb_pred]
unified["rf_correct"]  = unified["rf_pred"]  == unified["true_label"]
unified["xgb_correct"] = unified["xgb_pred"] == unified["true_label"]

llm_wins = unified[(unified["w4_nt_correct"] == True)  & (unified["xgb_correct"] == False)]
xgb_wins = unified[(unified["w4_nt_correct"] == False) & (unified["xgb_correct"] == True)]

print(f"30B beats XGBoost on: {len(llm_wins)} rows")
print(f"XGBoost beats 30B on: {len(xgb_wins)} rows")

print(f"\n=== WHERE 30B WINS - top occupations ===")
print(llm_wins["occupation"].value_counts().head(8))

print(f"\n=== WHERE XGBoost WINS - top occupations ===")
print(xgb_wins["occupation"].value_counts().head(8))

both_right = (unified["w4_nt_correct"] & unified["xgb_correct"]).sum()
both_wrong = (~unified["w4_nt_correct"] & ~unified["xgb_correct"]).sum()
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(
    [both_right, len(llm_wins), len(xgb_wins), both_wrong],
    labels=["Both right", "30B only", "XGBoost only", "Both wrong"],
    colors=["#1D9E75", "#378ADD", "#EF9F27", "#D85A30"],
    autopct="%1.1f%%", startangle=90
)
ax.set_title("30B Reasoning OFF vs XGBoost - Agreement")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_llm_vs_xgb.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_llm_vs_xgb.png")

## 9. Manager Removal Experiment
**Hypothesis:** Removing the most ambiguous occupation (manager) from the test set
should improve accuracy across all models since manager has mixed college/not_college
labels that confuse every model.

**Why manager:** In our K-Means cluster analysis, clusters dominated by manager had
consistently low few-shot accuracy. Manager is categorized as maybe in our audit
because it can require a degree or not depending on the industry.

**Method:** Filter out all manager rows, re-evaluate all models, compare to full results.

In [ ]:
# How many manager rows are in the test set?
manager_mask  = unified["occupation"].str.lower().str.strip() == "manager"
manager_count = manager_mask.sum()
manager_rows  = unified[manager_mask]

print(f"=== MANAGER ROW ANALYSIS ===")
print(f"Total test rows:    {len(unified)}")
print(f"Manager rows:       {manager_count} ({manager_count/len(unified)*100:.1f}%)")
print(f"Manager college rate: {(manager_rows['true_label'] == 'college').mean():.1%}")
print(f"\nManager true label distribution:")
print(manager_rows["true_label"].value_counts())
print()
print(f"Model accuracy ON manager rows:")
for name, col in [
    ("4B zero-shot",    "w3_zs_correct"),
    ("4B few-shot",     "w3_fs_correct"),
    ("30B zero-shot",   "w4_zs_correct"),
    ("30B few-shot",    "w4_fs_correct"),
    ("30B OFF",         "w4_nt_correct"),
    ("Random Forest",   "rf_correct"),
    ("XGBoost",         "xgb_correct"),
]:
    acc = manager_rows[col].mean()
    print(f"  {name:22s}: {acc:.1%}")

In [ ]:
# Create manager-free dataset
unified_no_mgr = unified[~manager_mask].reset_index(drop=True)

print(f"Test set WITHOUT manager: {len(unified_no_mgr)} rows")
print(f"Label balance: {unified_no_mgr['true_label'].value_counts().to_dict()}")
print()
print("=== ACCURACY COMPARISON: Full vs No-Manager Test Set ===")
print(f"{'Model':<26} {'Full':>8} {'No Mgr':>8} {'Change':>8}")
print("-" * 55)

model_cols = [
    ("4B zero-shot",       "w3_zs_correct"),
    ("4B few-shot",        "w3_fs_correct"),
    ("30B zero-shot",      "w4_zs_correct"),
    ("30B few-shot",       "w4_fs_correct"),
    ("30B reasoning OFF",  "w4_nt_correct"),
    ("Random Forest",      "rf_correct"),
    ("XGBoost",            "xgb_correct"),
]

comparison_rows = []
for name, col in model_cols:
    full_acc   = unified[col].mean()
    no_mgr_acc = unified_no_mgr[col].mean()
    change     = no_mgr_acc - full_acc
    print(f"  {name:<24} {full_acc:>7.1%} {no_mgr_acc:>7.1%} {change:>+8.1%}")
    comparison_rows.append({
        "model":      name,
        "full_acc":   round(full_acc, 4),
        "no_mgr_acc": round(no_mgr_acc, 4),
        "change":     round(change, 4)
    })

mgr_df = pd.DataFrame(comparison_rows)
mgr_df.to_csv(f"{RESULTS_DIR}/week5_manager_removal.csv", index=False)
print("\nSaved: results/week5_manager_removal.csv")

In [ ]:
# Visualization: full vs no-manager
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(mgr_df))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], mgr_df["full_acc"],
               width, label="Full test set", color="#378ADD", alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], mgr_df["no_mgr_acc"],
               width, label="Without manager", color="#1D9E75", alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(mgr_df["model"].values, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Accuracy")
ax.set_ylim(0.5, 0.9)
ax.set_title("Impact of Removing Manager Occupation from Test Set",
             fontsize=12, fontweight="bold")
ax.legend()

for i, row in mgr_df.iterrows():
    change = row["change"]
    color  = "#1D9E75" if change > 0 else "#D85A30"
    ax.text(i + width/2, row["no_mgr_acc"] + 0.005,
            f"{change:+.1%}", ha="center", fontsize=8, color=color, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_manager_removal.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_manager_removal.png")

In [ ]:
# Accuracy by verdict WITHOUT manager
print("=== ACCURACY BY VERDICT - Without Manager ===")
for verdict in ["yes", "no", "maybe"]:
    subset = unified_no_mgr[unified_no_mgr["audit_verdict"] == verdict]
    if len(subset) == 0:
        continue
    print(f"\n{verdict.upper()} ({len(subset)} rows):")
    for name, col in [
        ("4B few-shot",       "w3_fs_correct"),
        ("30B zero-shot",     "w4_zs_correct"),
        ("30B reasoning OFF", "w4_nt_correct"),
        ("XGBoost",           "xgb_correct"),
    ]:
        print(f"  {name}: {subset[col].mean():.1%}")

## 10. Final Summary Table - All Models

In [ ]:
summary = pd.DataFrame([
    {"model": "Random Forest",          **get_metrics(unified, "rf_pred"),    "week": 1},
    {"model": "XGBoost",                **get_metrics(unified, "xgb_pred"),   "week": 1},
    {"model": "Nano 4B zero-shot",      **get_metrics(unified, "w3_zs_pred"), "week": 3},
    {"model": "Nano 4B few-shot",       **get_metrics(unified, "w3_fs_pred"), "week": 3},
    {"model": "Nano 30B zero-shot",     **get_metrics(unified, "w4_zs_pred"), "week": 4},
    {"model": "Nano 30B few-shot",      **get_metrics(unified, "w4_fs_pred"), "week": 4},
    {"model": "Nano 30B reasoning OFF", **get_metrics(unified, "w4_nt_pred"), "week": 4},
])

print("=== FINAL SUMMARY TABLE (full test set) ===")
print(summary[["model", "accuracy", "macro_f1", "auc_roc", "n", "week"]].to_string(index=False))

summary.to_csv(f"{RESULTS_DIR}/week5_final_summary.csv", index=False)
print("\nSaved: results/week5_final_summary.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_list = ["accuracy", "macro_f1", "auc_roc"]
titles       = ["Accuracy", "Macro F1", "AUC-ROC"]
colors = ["#378ADD", "#1D9E75", "#EF9F27", "#D85A30", "#8B5CF6", "#EC4899", "#06B6D4"]

for ax, metric, title in zip(axes, metrics_list, titles):
    vals  = summary[metric].values
    names = [n.replace(" ", "\n") for n in summary["model"].values]
    bars  = ax.bar(names, vals, color=colors[:len(vals)])
    ax.set_title(title)
    ax.set_ylim(0.4, 1.0)
    ax.tick_params(axis="x", rotation=15, labelsize=7)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7)

plt.suptitle("Final Comparison: All Models", fontsize=11)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week5_final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week5_final_comparison.png")

## 11. Week 5 Summary

Fill in after running all cells:

**Agreement analysis:**
- All 5 models correct: ___ rows (___%)  
- All 5 models wrong: ___ rows (___%)  
- Mixed: ___ rows (___%)  

**Manager removal experiment:**
- Manager rows in test set: ___  
- Manager college rate: ___%  
- Manager accuracy across all models: ___% (below overall average)  
- Best accuracy improvement after removing manager: +___  
- Conclusion: ___  

**Feature importances:**
- Most important for RF: ___  
- Most important for XGBoost: ___  
- Nemotron reasoning traces: not available via Ollama (noted as limitation)  
